In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import json
import numpy as np
import joblib
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# =========================
# LOAD DATA
# =========================

print("Loading data...")

DATA_PATH = "/precomputed-features"   # <-- CHANGE THIS

X_train = np.load(f"{DATA_PATH}/X_train.npy")
y_train = np.load(f"{DATA_PATH}/y_train.npy")

X_test = np.load(f"{DATA_PATH}/X_test.npy")
y_test = np.load(f"{DATA_PATH}/y_test.npy")

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

# =========================
# SCALING
# =========================

print("Scaling features...")

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Save scaler
joblib.dump(scaler, "/feature_scaler.joblib")

# =========================
# MODELS
# =========================

models = {
   # "RF": RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42),
    "ET": ExtraTreesClassifier(n_estimators=100, n_jobs=-1, random_state=42),
    "GB": GradientBoostingClassifier(),
    #"SVM": LinearSVC(),
    #"LR": LogisticRegression(max_iter=1000, solver="saga", n_jobs=-1),
    #"NB": GaussianNB()
}

# =========================
# TRAIN & EVALUATE
# =========================

for name, model in models.items():
    print(f"\n🚀 Running {name}...")

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)

    print(f"{name} Accuracy: {acc}")

    # =========================
    # SAVE RESULTS
    # =========================

    save_dir = f"/output_data/{name}"
    os.makedirs(save_dir, exist_ok=True)

    # Accuracy
    with open(f"{save_dir}/accuracy.txt", "w") as f:
        f.write(f"Accuracy: {acc}")

    # Classification report
    with open(f"{save_dir}/classification_report.txt", "w") as f:
        f.write(report)

    # Confusion matrix JSON
    with open(f"{save_dir}/confusion_matrix.json", "w") as f:
        json.dump(cm.tolist(), f)

    # Save model
    joblib.dump(model, f"{save_dir}/{name}_model.joblib")

    # Plot confusion matrix
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap="Blues",
                xticklabels=["phishing", "legitimate"],
                yticklabels=["phishing", "legitimate"])
    plt.title(f"{name} Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    plt.savefig(f"{save_dir}/confusion_matrix.png")
    plt.close()

print("\n✅ All models completed!")

Loading data...
Train shape: (364199, 28)
Test shape: (51509, 28)
Scaling features...

🚀 Running ET...
ET Accuracy: 0.8928342619736357

🚀 Running GB...
GB Accuracy: 0.8592478984255179

✅ All models completed!
